# 90-Minute Lecture: Time-Series Analytics and Machine Learning for Bureau of Transportation Statistics (BTS) Datasets

**Datasets:**
- **DB28 — Air Carrier Traffic Statistics (monthly)**  
- **DB1B — Origin & Destination Survey (quarterly)**  
- **ASQP — Airline Service Quality Performance (monthly)**  

**Goal:** Provide a complete, classroom-ready walkthrough of:
- The four analytics layers (Descriptive, Diagnostic, Predictive, Prescriptive)  
- Classical time-series models (ARIMA / SARIMA / ARIMAX)  
- A simple neural-network model (N-BEATS)  
- Practical Google Colab demos for each dataset


## 1. Data Analytics (DA) and Machine Learning (ML) Foundations

### 1.1 What is Data Analytics?
**Data Analytics (DA)** is the systematic use of data to identify patterns, draw insights, and support evidence-based decisions.

### 1.2 The Four Types of Analytics
- **Descriptive Analytics:** What happened? (Historical performance)
- **Diagnostic Analytics:** Why did it happen? (Root cause)
- **Predictive Analytics:** What will happen next? (Forecasting)
- **Prescriptive Analytics:** What should we do? (Optimization / decision support)

### 1.3 Time-Series Concepts
- Level, trend, seasonality, noise
- Exogenous variables (population, GDP, fuel price, competition, weather)

### 1.4 Machine Learning (ML) Types
- Supervised learning: e.g., predict fare or delay
- Unsupervised learning: cluster routes or airports
- Reinforcement learning: adaptive pricing or scheduling


## 2. BTS Dataset Landscape

### 2.1 DB28 — Air Carrier Traffic Statistics (monthly)
- Granularity: carrier × origin × destination × month
- Fields: passengers, seats, distance, freight, mail
- Use: demand forecasting, capacity planning, route evaluation

### 2.2 DB1B — Origin & Destination Survey (quarterly)
- Granularity: 10% ticket sample
- Fields: fare, distance, coupons, itinerary
- Use: fare trends, elasticity, competitive analysis

### 2.3 ASQP — Airline Service Quality Performance (monthly, aggregated)
- Granularity: flight-level, aggregated to carrier × airport × month
- Fields: departure delay, arrival delay, cancellations, causes
- Use: reliability, on-time performance, delay drivers


## 3. Classical Forecasting and Neural Networks

### 3.1 ARIMA / SARIMA / ARIMAX
- ARIMA(p, d, q): autoregressive, differencing, moving average
- SARIMA(P, D, Q, s): seasonal extension (e.g., s = 12 for monthly)
- ARIMAX: ARIMA with exogenous variables

### 3.2 N-BEATS Neural Network
- Feed-forward neural network designed for time-series forecasting
- Learns trend and seasonality directly
- Simple to use via the `darts` Python library
- Good baseline for DB28, DB1B, and ASQP


In [ ]:
!pip install pandas numpy matplotlib darts[u] --quiet

## 4. Demo: DB28 — Monthly Passenger Forecast (Descriptive → Predictive → Prescriptive)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel

# Replace this with your real DB28 extract
db28 = pd.read_csv('/content/db28_sample.csv')
db28['Date'] = pd.to_datetime(db28[['Year', 'Month']].assign(Day=1))
db28 = db28.groupby('Date')['Passengers'].sum().reset_index()

# Descriptive
plt.figure(figsize=(10,4))
plt.plot(db28['Date'], db28['Passengers'])
plt.title('DB28 – Monthly Passengers')
plt.show()

# Predictive using N-BEATS
series = TimeSeries.from_dataframe(db28, 'Date', 'Passengers')
scaler = Scaler()
series_scaled = scaler.fit_transform(series)

train, test = series_scaled[:-12], series_scaled[-12:]

model = NBEATSModel(input_chunk_length=24, output_chunk_length=12, n_epochs=100, random_state=0)
model.fit(train)
forecast = model.predict(12)

series_scaled.plot(label='Actual')
forecast.plot(label='Forecast')
plt.legend(); plt.show()

# Simple prescriptive rule
growth = (forecast.values()[-1] - train.values()[-1]) / train.values()[-1]
if growth > 0.10:
    print('Recommendation: Increase capacity on key markets.')
else:
    print('Recommendation: Maintain or rebalance capacity.')


## 5. Demo: DB1B — Quarterly Fare Forecast

In [ ]:
# Replace with your cleaned DB1B quarterly summary
db1b = pd.read_csv('/content/db1b_quarterly.csv')
db1b['Date'] = pd.to_datetime(db1b['Quarter'])

series_db1b = TimeSeries.from_dataframe(db1b, 'Date', 'AvgFare')
train_db1b, test_db1b = series_db1b[:-4], series_db1b[-4:]

model_db1b = NBEATSModel(input_chunk_length=8, output_chunk_length=4, n_epochs=150, random_state=0)
model_db1b.fit(train_db1b)
forecast_db1b = model_db1b.predict(4)

series_db1b.plot(label='Actual')
forecast_db1b.plot(label='Forecast')
plt.legend(); plt.show()


## 6. Demo: ASQP — Monthly On-Time Performance Forecast

In [ ]:
# Replace with your aggregated ASQP data
asqp = pd.read_csv('/content/asqp_monthly.csv')
asqp['Date'] = pd.to_datetime(asqp[['Year', 'Month']].assign(Day=1))

series_asqp = TimeSeries.from_dataframe(asqp, 'Date', 'OTP')
train_asqp, test_asqp = series_asqp[:-12], series_asqp[-12:]

model_asqp = NBEATSModel(input_chunk_length=18, output_chunk_length=12, n_epochs=150, random_state=0)
model_asqp.fit(train_asqp)
forecast_asqp = model_asqp.predict(12)

series_asqp.plot(label='Actual')
forecast_asqp.plot(label='Forecast')
plt.legend(); plt.show()

if forecast_asqp.values().min() < 0.80:
    print('Prescriptive signal: add schedule padding / buffers to protect OTP.')



## 7. Questions for Each Dataset (Descriptive → Diagnostic → Predictive → Prescriptive)

### 7.1 DB28 — Air Carrier Traffic Statistics (monthly)

1. **Descriptive:** How have monthly passenger counts for a specific origin–destination pair evolved over the last 5 years?  
2. **Descriptive:** Which months consistently show the highest and lowest passenger demand for a given region or hub?  
3. **Diagnostic:** How did the entry or exit of a competing carrier on a route affect monthly passenger counts over the following year?  
4. **Predictive:** What is the forecasted passenger volume for a selected set of routes over the next 12 months, and what are the associated confidence intervals?  
5. **Prescriptive:** Given the forecasted demand and current seat capacity, which routes are likely to be over-capacity or under-capacity next season, and where should capacity be increased, decreased, or re-timed?  

### 7.2 DB1B — Origin & Destination Survey (quarterly)

1. **Descriptive:** How have average fares for a selected origin–destination market changed over the last 5–10 years?  
2. **Diagnostic:** How is the number of competitors serving a market related to changes in average fares and fare dispersion over time?  
3. **Diagnostic:** How do average fares differ across distance bands (short-haul vs. medium-haul vs. long-haul) and how stable are those differences?  
4. **Predictive:** What are the forecasted quarterly average fares for a given market over the next 4 quarters under current competitive conditions?  
5. **Prescriptive:** For which markets would a small fare reduction (e.g., 5–10%) likely yield the largest percentage gain in passenger volume, based on historical fare–demand relationships?  

### 7.3 ASQP — Airline Service Quality Performance (monthly)

1. **Descriptive:** What is the trend in monthly on-time performance (OTP) for a given carrier at a specific airport over the last 3–5 years?  
2. **Descriptive/Diagnostic:** Which delay causes (weather, carrier, national airspace system, security, late-arriving aircraft) dominate at a particular hub or region, and how have they changed seasonally?  
3. **Diagnostic:** How do changes in schedule density (flights per hour) relate to average delay minutes and cancellation rates at a given airport?  
4. **Predictive:** What is the forecasted OTP for a carrier–airport pair over the next 6–12 months, and how likely is it that OTP will fall below a target (e.g., 80%)?  
5. **Prescriptive:** At which airports or time-of-day windows should an airline add schedule padding, gate buffers, or staffing to keep OTP above target thresholds, given the predicted patterns?  

## 8. Cross-Dataset Questions (Combinations of DB28, DB1B, and ASQP)

1. **Fare–Demand Relationship:** How do changes in DB1B average fares for a market relate to DB28 passenger volumes on the same market, and can we quantify a price elasticity of demand that changes over time?  
2. **Reliability–Demand Interaction:** How does a sustained improvement or deterioration in ASQP on-time performance for a route or airport pair impact DB28 passenger volumes and DB1B fare sensitivity over subsequent seasons?  
3. **Capacity–Fare–Demand Alignment:** For markets where DB28 shows rising demand and DB1B shows stable or increasing fares, what capacity adjustments (seats, frequency) are most likely to maximize revenue while maintaining acceptable ASQP reliability?  
4. **Network Planning:** By combining DB28 demand forecasts, DB1B fare projections, and ASQP reliability patterns, which new routes or frequency increases should be prioritized to balance profitability, operational robustness, and customer experience?  
5. **Scenario Analysis:** Under a scenario of increasing fuel prices and tighter OTP requirements, how do different combinations of capacity changes (DB28), fare adjustments (DB1B), and schedule padding (ASQP) affect expected revenue, load factor, and on-time performance across the network?  

*End of 90-Minute Lecture Notebook.*
